Energy Balance Assumptions

Steps 1 & 2 (Acylation & Enamine Formation): The first stage of this synthesis involves an acyl chloride reacting with an acrylate derivative, followed by an amine exchange. Acyl chloride substitutions are notoriously exothermic due to the high energy of the C-Cl bond compared to the resulting C-C or C-N bonds. Standard Schotten-Baumann-type acylations release between $-70 \text{ to } -100\text{ kJ/mol}$. We will use a calculated estimate of $-80\text{ kJ/mol}$.

Step 4 (S_NAr / Ring Closure with Piperazine): We can accurately estimate this using standard Bond Dissociation Energies (BDEs).<br>Bonds Broken: Aryl C–F bond ($\sim 523\text{ kJ/mol}$) + Piperazine N–H bond ($\sim 390\text{ kJ/mol}$) = $913\text{ kJ/mol}$ required.
<br>Bonds Formed: Aryl C–N bond ($\sim 430\text{ kJ/mol}$) + H–F bond ($\sim 570\text{ kJ/mol}$) = $1000\text{ kJ/mol}$ released.
<br>Estimated $\Delta H_{rxn}$ = $913 - 1000 =$ $-87\text{ kJ/mol}$.

Step 5 (Base-Catalyzed Ester Hydrolysis): The final upstream step uses TBAOH to hydrolyze the ethyl ester. Base-catalyzed saponification is a textbook thermodynamic value. The literature standard for the saponification of an ethyl ester with a strong base yields a $\Delta H$ of exactly $-48 \text{ to } -52\text{ kJ/mol}$. We will lock this in at $-50\text{ kJ/mol}$.

Fluid Properties ($C_p$ and $\rho$): The system uses a mixture of DMSO and 2-MeTHF. The specific heat capacity ($C_p$) of pure DMSO is $1.96\text{ J/(g}\cdot\text{K)}$ and 2-MeTHF is $\sim 1.9\text{ J/(g}\cdot\text{K)}$. The density of DMSO is $1.10\text{ g/mL}$ and 2-MeTHF is $0.85\text{ g/mL}$. A mixed process stream is therefore safely estimated at $C_p = 1.95\text{ J/(g}\cdot\text{K)}$ and $\rho = 0.95\text{ g/mL}$.

In [ ]:
import math
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML

# ==================================================== #
#                 USER CONFIGURATION                   #
# ==================================================== #

# --- EXPORT SETTINGS ---
EXPORT_TO_EXCEL = True
EXCEL_PREFIX = 'Cipro_Mass_Balance'

# --- MODE AND TARGETS ---
CALCULATION_MODE = 'backward' # 'forward' (specify feeds) or 'backward' (specify target pure API)
feed1_molar_flow_mol_h = 0.225 # Feed 1 Acrylate molar flow rate, mol/h (used if mode is 'forward')
target_pure_API_kg_h = 6.875  # Target Pure API Production, kg/h (used if mode is 'backward')

# --- PROCESS YIELDS & SOLVENT RATIOS ---
YIELDS = {
    'steps_1_and_2': 0.91,
    'extraction_CLLE': 0.88,
    'DMA': 0.05, ### I'm assuming here based on the purity tests that any impurities in step 2 are due to DMA and rounding that to 5%
    'steps_4_and_5': 0.90,
    'downstream_purification': 0.309
}

# Solvents scaled to Liters and Kilograms (L/kg equivalent to mL/g)
SOLVENT_RATIOS = {
    'T1_precip_agent_vol_ratio': 0.58,
    'T3_wash1_acetone_L_kg': 20.0,
    'T4_wash2_water_L_kg': 20.0,
    'T5_dissolution_L_kg': 16.5,
    'T6_antisolvent_vol_ratio': 1.5,
    'T7_wash_final_L_kg': 8.0
}

# --- THERMODYNAMICS & FLUID PROPERTIES ---
THERMO_PARAMS = {
    'dH_step1_2': -80.0,  # kJ/mol
    'dH_step4': -87.0,    # kJ/mol
    'dH_step5': -50.0,    # kJ/mol
    'density_kg_L': 0.95, # kg/L (equivalent to g/mL)
    'Cp_kJ_kgK': 1.95,    # kJ/kg*K (equivalent to J/g*K)
    'T_ambient': 25.0,    # Celsius
    'T_reactor_1': 150.0  # Celsius
}

# --- REACTOR KINETICS & FEED MOLARITIES ---
RESIDENCE_TIMES_TAU = {
    'Reactor_1_Step_1': 2.0,   # Acylation at 150 C (minutes)
    'Reactor_2_Step_2': 1.3,   # Amine substitution at ambient temp (minutes)
    'Reactor_4_Step_4': 5.0,   # Ring closure (SNAr) at 150 C (minutes)
    'Reactor_5_Step_5': 3.2    # Ester hydrolysis at 150 C (minutes)
}

FEED_MOLARITIES = {
    'Feed_1_Acrylate': 1.0,   # mol/L
    'Feed_2_Amine': 1.25,     # mol/L
    'Feed_3_Piperazine': 2.3, # mol/L
    'Feed_4_TBAOH': 1.5       # mol/L
}


# ==================================================== #
#                  CORE CALCULATION CLASSES            #
# ==================================================== #

class CiprofloxacinMassEnergyBalance:
    def __init__(self, mode, yields, ratios, thermo, target_kg_h, feed_mol_h):
        self.mode = mode
        self.yields = yields
        self.ratios = ratios
        self.thermo = thermo

        self.MW_cipro = 331.34 # g/mol
        self.MW_cipro_hcl_hydrate = 385.8 # g/mol

        self.base_feed1_mol_h = 0.225 # Baseline from 3.75 mL/min * 1.0 mol/L
        self.ratio_feed2 = 1.0
        self.ratio_crude_stream = 14.2 / 3.75

        self.ratio_v_dot_step1 = 7.50 / 3.75
        self.ratio_v_dot_step2 = 7.50 / 3.75
        self.ratio_v_dot_step4 = 10.85 / 3.75
        self.ratio_v_dot_step5 = 14.20 / 3.75

        self.theoretical_baseline_crude_kg_h = 0.05373 # Converted to kg/h
        self.empirical_correction_factor = 0.0290 / self.theoretical_baseline_crude_kg_h
        self.unc = 0.088

        if self.mode == 'forward':
            self.feed1_mol_h = feed_mol_h
        elif self.mode == 'backward':
            baseline_pure_kg_h = 0.01043 # Converted to kg/h
            scale_factor = target_kg_h / baseline_pure_kg_h
            self.feed1_mol_h = self.base_feed1_mol_h * scale_factor
        else:
            raise ValueError("Mode must be 'forward' or 'backward'")

        # Volumetric baseline for ratios
        self.feed1_flow_L_h = self.feed1_mol_h / FEED_MOLARITIES['Feed_1_Acrylate']

    def calculate(self):
        molar_1_acrylate_mol_h = self.feed1_mol_h

        # Molar ratios based on volumes
        feed2_flow_L_h = self.feed1_flow_L_h * self.ratio_feed2
        molar_4_amine_mol_h = feed2_flow_L_h * FEED_MOLARITIES['Feed_2_Amine']
        limiting_molar_mol_h = min(molar_1_acrylate_mol_h, molar_4_amine_mol_h)

        molar_1_acyl_chloride = limiting_molar_mol_h
        molar_3_keto_ester = limiting_molar_mol_h * 1.0
        molar_5_enamine = limiting_molar_mol_h * self.yields['steps_1_and_2']
        molar_6_dma = self.yields['DMA'] * molar_5_enamine
        molar_5_extracted = molar_5_enamine * self.yields['extraction_CLLE']
        molar_8_piperazine = molar_5_extracted
        molar_9_cipro_ester = molar_5_extracted * (self.yields['steps_4_and_5'] ** 0.5)
        molar_10_tbaoh = molar_9_cipro_ester

        molar_11_crude_theo = molar_5_extracted * self.yields['steps_4_and_5']
        molar_11_crude_actual = molar_11_crude_theo * self.empirical_correction_factor
        mass_crude_kg_h = (molar_11_crude_actual * self.MW_cipro) / 1000

        molar_pure = molar_11_crude_actual * self.yields['downstream_purification']
        mass_pure_kg_h = (molar_pure * self.MW_cipro_hcl_hydrate) / 1000

        crude_stream_vol_L_h = self.feed1_flow_L_h * self.ratio_crude_stream
        t1_precip = crude_stream_vol_L_h * self.ratios['T1_precip_agent_vol_ratio']
        t3_wash1 = mass_crude_kg_h * self.ratios['T3_wash1_acetone_L_kg']
        t4_wash2 = mass_crude_kg_h * self.ratios['T4_wash2_water_L_kg']
        t5_dissolve = mass_crude_kg_h * self.ratios['T5_dissolution_L_kg']
        t6_antisolvent = (crude_stream_vol_L_h + t1_precip) * self.ratios['T6_antisolvent_vol_ratio']
        t7_final_wash = mass_pure_kg_h * self.ratios['T7_wash_final_L_kg']

        v_dot_reactors_L_h = {
            'Reactor_1_Step_1': self.feed1_flow_L_h * self.ratio_v_dot_step1,
            'Reactor_2_Step_2': self.feed1_flow_L_h * self.ratio_v_dot_step2,
            'Reactor_4_Step_4': self.feed1_flow_L_h * self.ratio_v_dot_step4,
            'Reactor_5_Step_5': self.feed1_flow_L_h * self.ratio_v_dot_step5
        }

        # Energy Conversions (mol/h to mol/s, then kW)
        mol_s_step1 = limiting_molar_mol_h / 3600.0
        mol_s_step4 = molar_5_extracted / 3600.0
        mol_s_step5 = molar_9_cipro_ester / 3600.0

        q_rxn_1_2_kW = mol_s_step1 * self.thermo['dH_step1_2']
        q_rxn_4_kW = mol_s_step4 * self.thermo['dH_step4']
        q_rxn_5_kW = mol_s_step5 * self.thermo['dH_step5']
        total_reaction_heat_kW = q_rxn_1_2_kW + q_rxn_4_kW + q_rxn_5_kW

        vol_s_feed_stream_L_s = (self.feed1_flow_L_h + feed2_flow_L_h) / 3600.0
        mass_s_feed_stream_kg_s = vol_s_feed_stream_L_s * self.thermo['density_kg_L']

        dT_heat = self.thermo['T_reactor_1'] - self.thermo['T_ambient']
        q_sensible_heating_kW = mass_s_feed_stream_kg_s * self.thermo['Cp_kJ_kgK'] * dT_heat

        dT_cool = self.thermo['T_ambient'] - self.thermo['T_reactor_1']
        q_sensible_cooling_kW = mass_s_feed_stream_kg_s * self.thermo['Cp_kJ_kgK'] * dT_cool

        u = lambda val: (val, val * self.unc)

        return {
            "components": {
                "1 - Acyl Chloride": u(molar_1_acyl_chloride),
                "2 - Acrylate SM": u(molar_1_acrylate_mol_h),
                "3 - Keto-ester": u(molar_3_keto_ester),
                "4 - Cyclopropylamine": u(molar_4_amine_mol_h),
                "5 - Cyclopropyl-enamine": u(molar_5_enamine),
                "6 - Dimethylamine": u(molar_6_dma),
                "8 - Piperazine": u(molar_8_piperazine),
                "9 - Cipro-ester": u(molar_9_cipro_ester),
                "10 - TBAOH": u(molar_10_tbaoh),
                "11 - Crude Ciprofloxacin": u(molar_11_crude_actual),
            },
            "solvents": {
                "T1 - Precip Agent": u(t1_precip),
                "T3 - Wash 1 (Acetone)": u(t3_wash1),
                "T4 - Wash 2 (Water)": u(t4_wash2),
                "T5 - Dissolution (HCl)": u(t5_dissolve),
                "T6 - Isopropanol Antisolvent": u(t6_antisolvent),
                "T7 - Wash Solvent (Final)": u(t7_final_wash),
            },
            "metrics": {
                "Crude API Rate (kg/h)": u(mass_crude_kg_h),
                "Pure API Rate (kg/h)": u(mass_pure_kg_h),
                "q_rxn_1_2 (kW)": q_rxn_1_2_kW,
                "q_rxn_4 (kW)": q_rxn_4_kW,
                "q_rxn_5 (kW)": q_rxn_5_kW,
                "q_rxn_total (kW)": total_reaction_heat_kW,
                "q_sensible_heat (kW)": q_sensible_heating_kW,
                "q_sensible_cool (kW)": q_sensible_cooling_kW,
            },
            "v_dots": v_dot_reactors_L_h,
            "raw_n_dots": {
                'acrylate': molar_1_acrylate_mol_h,
                'amine': molar_4_amine_mol_h,
                'piperazine': molar_8_piperazine,
                'tbaoh': molar_10_tbaoh
            }
        }

class PFRVolumeCalculator:
    def __init__(self, tau_min, v_dot_L_h):
        self.tau_min = tau_min
        self.v_dot_L_h = v_dot_L_h

    def calculate_volumes(self):
        volumes = {}
        for reactor in self.tau_min.keys():
            # Convert tau from minutes to hours for volume calc: V = v_dot * tau
            tau_hr = self.tau_min[reactor] / 60.0
            v_L = self.v_dot_L_h[reactor] * tau_hr
            volumes[reactor] = {'Volume (L)': v_L, 'v_dot (L/h)': self.v_dot_L_h[reactor], 'tau (hours)': tau_hr}
        return volumes

class ContinuousSpeciesTracker:
    def __init__(self, feed_molarities, req_n_dots, yields):
        self.molarities = feed_molarities
        self.yield_1_2 = yields['steps_1_and_2']
        self.DMA = yields['DMA']
        self.yield_clle = yields['extraction_CLLE']
        self.yield_4 = math.sqrt(yields['steps_4_and_5'])
        self.yield_5 = math.sqrt(yields['steps_4_and_5'])

        self.req_n_dot_acrylate = req_n_dots['acrylate']
        self.req_n_dot_amine = req_n_dots['amine']
        self.req_n_dot_piperazine = req_n_dots['piperazine']
        self.req_n_dot_tbaoh = req_n_dots['tbaoh']

        self.v_dot_feeds = {
            'Feed_1_Acrylate': self.req_n_dot_acrylate / self.molarities['Feed_1_Acrylate'],
            'Feed_2_Amine': self.req_n_dot_amine / self.molarities['Feed_2_Amine'],
            'Feed_3_Piperazine': self.req_n_dot_piperazine / self.molarities['Feed_3_Piperazine'],
            'Feed_4_TBAOH': self.req_n_dot_tbaoh / self.molarities['Feed_4_TBAOH']
        }

        self.stream = {
            '2_Acrylate': 0.0, '4_Amine': 0.0, '5_Enamine': 0.0, '6_DMA': 0.0,
            '8_Piperazine': 0.0, '9_Cipro_Ester': 0.0, '10_TBAOH': 0.0, '11_Crude_Cipro': 0.0
        }
        self.v_dot_total = 0.0
        self.report = {}

    def run_process(self):
        # 1. REACTOR 1 & 2
        self.v_dot_total = self.v_dot_feeds['Feed_1_Acrylate'] + self.v_dot_feeds['Feed_2_Amine']
        n_dot_reacted = self.req_n_dot_acrylate * self.yield_1_2
        self.stream['2_Acrylate'] = self.req_n_dot_acrylate - n_dot_reacted
        self.stream['4_Amine'] = self.req_n_dot_amine - n_dot_reacted
        self.stream['5_Enamine'] = n_dot_reacted
        self.stream['6_DMA'] = self.DMA * n_dot_reacted
        self.report['1_Acylation_Enamine'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        # 2. CLLE
        self.stream['5_Enamine'] = self.stream['5_Enamine'] * self.yield_clle
        self.stream['6_DMA'] = 0.0
        self.stream['4_Amine'] = 0.0
        self.report['2_CLLE_Organic'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        # 3. REACTOR 4
        self.v_dot_total += self.v_dot_feeds['Feed_3_Piperazine']
        self.stream['8_Piperazine'] += self.req_n_dot_piperazine
        n_dot_reacted_4 = self.stream['5_Enamine'] * self.yield_4
        self.stream['5_Enamine'] -= n_dot_reacted_4
        self.stream['8_Piperazine'] -= n_dot_reacted_4
        self.stream['9_Cipro_Ester'] += n_dot_reacted_4
        self.report['3_Ring_Closure'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        # 4. REACTOR 5
        self.v_dot_total += self.v_dot_feeds['Feed_4_TBAOH']
        self.stream['10_TBAOH'] += self.req_n_dot_tbaoh
        n_dot_reacted_5 = self.stream['9_Cipro_Ester'] * self.yield_5
        self.stream['9_Cipro_Ester'] -= n_dot_reacted_5
        self.stream['10_TBAOH'] -= n_dot_reacted_5
        self.stream['11_Crude_Cipro'] += n_dot_reacted_5
        self.report['4_Hydrolysis'] = {'v_dot': self.v_dot_total, 'stream': self.stream.copy()}

        return self.report

# ==================================================== #
#                  EXECUTION & REPORTING               #
# ==================================================== #

def generate_reports():
    # 1. Run Calculations
    balance = CiprofloxacinMassEnergyBalance(CALCULATION_MODE, YIELDS, SOLVENT_RATIOS, THERMO_PARAMS, target_pure_API_kg_h, feed1_molar_flow_mol_h)
    res = balance.calculate()

    pfr_calc = PFRVolumeCalculator(RESIDENCE_TIMES_TAU, res['v_dots'])
    volumes = pfr_calc.calculate_volumes()

    tracker = ContinuousSpeciesTracker(FEED_MOLARITIES, res['raw_n_dots'], YIELDS)
    tracker_report = tracker.run_process()

    # 2. Format Pandas DataFrames
    pd.options.display.float_format = '{:,.3f}'.format

    df_comp = pd.DataFrame.from_dict(res['components'], orient='index', columns=['Value', 'Uncertainty (±)'])
    df_comp.index.name = 'Upstream Molar Flows (mol/h)'

    df_solv = pd.DataFrame.from_dict(res['solvents'], orient='index', columns=['Value', 'Uncertainty (±)'])
    df_solv.index.name = 'Downstream Volumetric Flows (L/h)'

    # Store daily target for naming purposes
    daily_production_kg = res['metrics']['Pure API Rate (kg/h)'][0] * 24

    metrics = [
        ("Crude API Mass Rate (kg/h)", res['metrics']['Crude API Rate (kg/h)'][0]),
        ("Pure API Mass Rate (kg/h)", res['metrics']['Pure API Rate (kg/h)'][0]),
        ("Pre-Heating Feed Duty (kW)", res['metrics']['q_sensible_heat (kW)']),
        ("Quenching Duty (kW)", res['metrics']['q_sensible_cool (kW)']),
        ("Total Isothermal Rxn Load (kW)", abs(res['metrics']['q_rxn_total (kW)']))
    ]
    df_metrics = pd.DataFrame(metrics, columns=['Production & Thermal Duties', 'Value']).set_index('Production & Thermal Duties')

    df_vols = pd.DataFrame.from_dict(volumes, orient='index')
    df_vols.index.name = 'Required PFR Tubular Reactor Volumes'

    # Restructure Tracker data for a clean dataframe
    tracker_data = {}
    for stage, data in tracker_report.items():
        row = {'Stream Total v_dot (L/h)': data['v_dot']}
        # Keep active components > 0.001 mol/h for cleanliness
        row.update({k + " (mol/h)": v for k, v in data['stream'].items() if v > 0.001})
        tracker_data[stage] = row

    df_tracker = pd.DataFrame.from_dict(tracker_data, orient='index').fillna(0)
    df_tracker.index.name = 'Continuous Species Tracker (Molar & Volumetric Flows)'

    # 3. Export to Excel (if enabled)
    if EXPORT_TO_EXCEL:
        date_str = datetime.now().strftime("%m-%d-%y")
        # Example format: Cipro_Mass_Balance_10-25-23_95040kg.xlsx
        dynamic_filename = f"{EXCEL_PREFIX}_{date_str}_{daily_production_kg:g}kg.xlsx"

        with pd.ExcelWriter(dynamic_filename) as writer:
            df_comp.to_excel(writer, sheet_name='Upstream_Flows')
            df_solv.to_excel(writer, sheet_name='Downstream_Flows')
            df_metrics.to_excel(writer, sheet_name='Metrics')
            df_vols.to_excel(writer, sheet_name='Reactor_Volumes')
            df_tracker.to_excel(writer, sheet_name='Species_Tracker')
        print(f"✅ Successfully exported all dataframes to: {dynamic_filename}")

    # 4. Display Results in Jupyter
    display(HTML(f"<h3>Continuous Ciprofloxacin Balance ({CALCULATION_MODE.upper()} MODE)</h3>"))
    display(df_comp)
    display(df_solv)
    display(df_metrics)
    display(df_vols)
    display(df_tracker)

if __name__ == "__main__":
    generate_reports()

✅ Successfully exported all dataframes to: Cipro_Mass_Balance_03-18-26_165.063kg.xlsx


,Value,Uncertainty (±)
Upstream Molar Flows (mol/h),,
1 - Acyl Chloride,148.310,13.051
2 - Acrylate SM,148.310,13.051
3 - Keto-ester,148.310,13.051
4 - Cyclopropylamine,185.388,16.314
5 - Cyclopropyl-enamine,134.962,11.877
6 - Dimethylamine,6.748,0.594
8 - Piperazine,118.767,10.451
9 - Cipro-ester,112.672,9.915
10 - TBAOH,112.672,9.915


,Value,Uncertainty (±)
Downstream Volumetric Flows (L/h),,
T1 - Precip Agent,325.729,28.664
T3 - Wash 1 (Acetone),382.316,33.644
T4 - Wash 2 (Water),382.316,33.644
T5 - Dissolution (HCl),315.411,27.756
T6 - Isopropanol Antisolvent,"1,330.995",117.128
T7 - Wash Solvent (Final),55.021,4.842


,Value
Production & Thermal Duties,
Crude API Mass Rate (kg/h),19.116
Pure API Mass Rate (kg/h),6.878
Pre-Heating Feed Duty (kW),19.079
Quenching Duty (kW),-19.079
Total Isothermal Rxn Load (kW),7.731


,Volume (L),v_dot (L/h),tau (hours)
Required PFR Tubular Reactor Volumes,,,
Reactor_1_Step_1,9.887,296.620,0.033
Reactor_2_Step_2,6.427,296.620,0.022
Reactor_4_Step_4,35.759,429.111,0.083
Reactor_5_Step_5,29.952,561.601,0.053


,Stream Total v_dot (L/h),2_Acrylate (mol/h),4_Amine (mol/h),5_Enamine (mol/h),6_DMA (mol/h),8_Piperazine (mol/h),9_Cipro_Ester (mol/h),10_TBAOH (mol/h),11_Crude_Cipro (mol/h)
Continuous Species Tracker (Molar & Volumetric Flows),,,,,,,,,
1_Acylation_Enamine,296.620,13.348,50.425,134.962,6.748,0.000,0.000,0.000,0.000
2_CLLE_Organic,296.620,13.348,0.000,118.767,0.000,0.000,0.000,0.000,0.000
3_Ring_Closure,348.258,13.348,0.000,6.095,0.000,6.095,112.672,0.000,0.000
4_Hydrolysis,423.373,13.348,0.000,6.095,0.000,6.095,5.782,5.782,106.890
